# Your first threshold curve — with the real research stack

Companion notebook to **Lattice Atlas → The Surface Code Lab**.

In the Lab you decoded a *code-capacity* toy model (perfect measurements) and watched the curves cross near p ≈ 15%. Here you run the same experiment the way working researchers do — with **Stim** (noisy syndrome-extraction circuits, the simulator behind Google's below-threshold papers) and **PyMatching** (minimum-weight perfect matching) — and watch the threshold drop to the famous **~1%**.

Prerequisite topics: *surface code*, *syndrome extraction circuits*, *decoding/MWPM*, *fault tolerance & thresholds*.

In [ ]:
%pip install -q stim pymatching matplotlib numpy

## 1 · Build a noisy surface-code memory circuit

`stim.Circuit.generated` produces the standard rotated memory-Z experiment — the same circuit family as the Lab's **Download .stim** button, with circuit-level noise: every gate, reset, and measurement can fail.

In [ ]:
import stim
import pymatching
import numpy as np

circuit = stim.Circuit.generated(
    "surface_code:rotated_memory_z",
    distance=3,
    rounds=3,
    after_clifford_depolarization=0.005,
    after_reset_flip_probability=0.005,
    before_measure_flip_probability=0.005,
    before_round_data_depolarization=0.005,
)
print(f"{circuit.num_qubits} qubits, {circuit.num_detectors} detectors, {circuit.num_observables} observable")

## 2 · Sample and decode

The pipeline every decoding paper uses:

1. The circuit's **detector error model** (DEM) lists every possible fault and which detectors it flips — this is the decoder's map of the world.
2. PyMatching builds its matching graph straight from the DEM.
3. Stim samples detection events; PyMatching predicts the logical observable; disagreements are **logical errors**.

In [ ]:
def logical_error_rate(distance: int, p: float, shots: int) -> float:
    circuit = stim.Circuit.generated(
        "surface_code:rotated_memory_z",
        distance=distance,
        rounds=distance,
        after_clifford_depolarization=p,
        after_reset_flip_probability=p,
        before_measure_flip_probability=p,
        before_round_data_depolarization=p,
    )
    dem = circuit.detector_error_model(decompose_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(dem)
    sampler = circuit.compile_detector_sampler()
    detections, observables = sampler.sample(shots, separate_observables=True)
    predictions = matcher.decode_batch(detections)
    errors = np.sum(np.any(predictions != observables, axis=1))
    return errors / shots

print("d=3, p=0.5% →", logical_error_rate(3, 0.005, 5000))

## 3 · The sweep

Same experiment as the Lab's chart, now with noisy measurements. Expect the crossing near **p ≈ 1%** instead of 15% — measurement noise is exactly what moves it.

(A few minutes at these shot counts; lower `SHOTS` for a quick look.)

In [ ]:
import matplotlib.pyplot as plt

PS = [0.002, 0.004, 0.006, 0.008, 0.010, 0.012, 0.015]
DISTANCES = [3, 5, 7]
SHOTS = 20_000

results = {d: [logical_error_rate(d, p, SHOTS) for p in PS] for d in DISTANCES}

fig, ax = plt.subplots(figsize=(7, 5))
for d, color in zip(DISTANCES, ["#0891B2", "#8B5CF6", "#D97706"]):
    rates = results[d]
    err = [1.96 * np.sqrt(r * (1 - r) / SHOTS) if r > 0 else 0 for r in rates]
    ax.errorbar(PS, rates, yerr=err, marker="o", color=color, label=f"d={d}", capsize=3)
ax.set_yscale("log")
ax.set_xlabel("physical error rate p")
ax.set_ylabel("logical error rate")
ax.set_title("Rotated surface code memory, circuit-level noise (Stim + PyMatching)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 4 · Read your plot like the papers do

- **Below the crossing**: bigger d, lower logical error — the below-threshold regime. Compute Λ = ε(d)/ε(d+2) at p = 0.4% and compare with Google's measured Λ ≈ 2.1 (arXiv:2408.13687).
- **The crossing** is the circuit-level threshold, ≈ 0.7–1% for this noise family — the "famous one percent" that set hardware targets for two decades.

### Keep going
- Load your own Lab export: `stim.Circuit(open("surface_code_d5_p0.080.stim").read())` and decode it with the same pipeline.
- Swap in `"surface_code:rotated_memory_x"` — why are the curves (almost) the same?
- Try `rounds=1` vs `rounds=d` — watch measurement noise destroy the single-round code and rediscover *why* syndromes are repeated.
- For serious sweeps, use `sinter` (Stim's Monte-Carlo harness) — it parallelizes and handles statistics properly.
- Paste any circuit into **Crumble** (https://algassert.com/crumble) to step through it tick by tick.